In [1]:
import pandas as pd
import numpy as np
df = pd.read_csv('UpdatedResumeDataSet.csv')
df.head()

,Category,Resume
0,Data Science,Skills * Programming Languages: Python (pandas...
1,Data Science,Education Details \r\nMay 2013 to May 2017 B.E...
2,Data Science,"Areas of Interest Deep Learning, Control Syste..."
3,Data Science,Skills â¢ R â¢ Python â¢ SAP HANA â¢ Table...
4,Data Science,"Education Details \r\n MCA YMCAUST, Faridab..."


In [12]:
# Merge tool-based & overlapping roles
df["Category"] = df["Category"].replace({
    "Hadoop": "ETL Developer",
    "Automation Testing": "Testing",
    "Database": "ETL Developer",
    "PMO": "Operations Manager"
})

# Remove weak / noisy labels
df = df[~df["Category"].isin([
    "Arts",
    "Sales"
])]


In [35]:
df['Category'].value_counts()

Category
ETL Developer                115
Testing                       96
Java Developer                84
Operations Manager            70
DevOps Engineer               55
Python Developer              48
Web Designing                 45
HR                            43
Data Science                  40
Blockchain                    40
Mechanical Engineer           40
Health and fitness            30
Electrical Engineering        30
DotNet Developer              28
Business Analyst              28
Network Security Engineer     25
SAP Developer                 24
Civil Engineer                24
Advocate                      20
Name: count, dtype: int64

In [36]:
df['Resume'] = df['Resume'].str.lower()
df['Resume'] = df['Resume'].str.strip()
df['Resume'] = df['Resume'].str.replace(r'\s+', ' ', regex=True) #removing extra spaces 

In [37]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        ngram_range=(1, 2),
        max_df=0.85,
        min_df=2
    )),
    ("clf", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        solver="lbfgs",
        multi_class="auto"
    ))
])

In [38]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
X = df['Resume']
le = LabelEncoder()
y = df['Category']
y = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [39]:
pipeline.fit(X_train, y_train)

c:\Users\amogh\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


,steps,"[('tfidf', ...), ('clf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,None


In [40]:
def guess_top5_jobs(resume):
    pred = pipeline.predict_proba(resume)
    pred = pred.ravel()
    # Use argsort to get indices that would sort the array in ascending order
    sorted_indices_asc = np.argsort(pred)
    # To get the top (largest) values, reverse the array of indices and slice the first 'num_elements'
    top_indices = sorted_indices_asc[::-1][:5]
    return le.inverse_transform(top_indices)

In [41]:
sample_resume = ["""PROFESSIONAL SUMMARY
Detail-oriented Test Engineer with 5+ years of experience in Software Quality Assurance, specializing in automated testing frameworks, API testing, and performance analysis. Proven track record of increasing testing efficiency by 70% and reducing post-release defects by 30% through robust automation and Agile methodologies. 
TECHNICAL SKILLS
Testing Tools: Selenium WebDriver, Cucumber, JUnit, TestNG, Postman
Languages: Java, Python, SQL
CI/CD & DevOps: Jenkins, Azure Pipelines, Git
Test Management: Jira, Zephyr, TestRail
Methodologies: Agile/Scrum, Functional Testing, Regression Testing, API Testing 
PROFESSIONAL EXPERIENCE
[Company Name] | [Location]
QA Test Engineer | [Start Date] – [Present] 
Reduced manual testing effort by 80% by implementing automated regression testing scripts using Selenium WebDriver and Java.
Collaborated with developers in Agile sprints to resolve 25+ critical defects, ensuring a 98% test case pass rate.
Designed and executed API tests using Postman, reducing API-related defects by 20%.
Integrated automated test suites into Jenkins CI/CD pipeline, decreasing deployment time by 35%. 
[Previous Company Name] | [Location]
Junior Test Engineer | [Start Date] – [End Date] 
Executed 500+ manual test cases per release, identifying and reporting 50+ defects.
Developed comprehensive test plans and scenarios that increased test coverage from 0% to 40%.
Utilized Jira for tracking bugs, leading to a 30% reduction in bug-reporting errors. 
EDUCATION
Bachelor of Science in Computer Science
[University Name], [City, State]
CERTIFICATIONS
ISTQB Certified Tester, Foundation Level (CTFL) """]

top5_jobs = guess_top5_jobs(sample_resume)
top5_jobs

array(['Testing', 'Business Analyst', 'ETL Developer', 'Java Developer',
       'Python Developer'], dtype=object)

In [42]:
sample_resume_1 = ["""Experienced technology professional with a background in building reliable, scalable, and data-driven systems across multiple environments. Strong focus on system quality, automation, and continuous improvement throughout the software lifecycle. Adept at collaborating with engineering, analytics, and product teams to deliver stable and measurable outcomes.

Hands-on experience designing automated workflows for validating data integrity, service reliability, and system performance. Regularly works with structured datasets, writing optimized queries and developing scripts to monitor trends, detect anomalies, and generate actionable insights. Comfortable working with large datasets and distributed systems concepts.

Contributed to the development and maintenance of backend services, ensuring consistent behavior across environments. Implemented automated checks for service responses, data accuracy, and deployment readiness. Experience integrating validation processes into continuous integration pipelines to reduce manual intervention and improve release confidence.

Worked closely with stakeholders to understand functional requirements, translate them into testable scenarios, and verify end-to-end behavior of applications. Participated in sprint planning, requirement reviews, and post-release analysis to identify gaps and optimize workflows.

Experience with cloud-based platforms for deploying applications, managing resources, and monitoring system health. Familiar with containerization, version control systems, and infrastructure automation concepts. Actively involved in incident analysis, root cause identification, and performance optimization.

Strong analytical mindset with attention to detail and process improvement. Interested in roles that combine software engineering, data analysis, and automation to improve system reliability and business outcomes."""]
top5_jobs = guess_top5_jobs(sample_resume_1)
top5_jobs

array(['ETL Developer', 'Operations Manager', 'Testing', 'Data Science',
       'Business Analyst'], dtype=object)

In [43]:
sample_resume_2 = ["""Experienced professional with a background in developing interactive web applications and analyzing data to support business decisions. Skilled in translating data insights into user-friendly interfaces and actionable reports. Strong understanding of both frontend development and analytical problem-solving.

Hands-on experience building responsive web applications using modern JavaScript frameworks, implementing clean UI components, and integrating backend APIs. Comfortable working with HTML, CSS, and JavaScript to create accessible, mobile-friendly user experiences. Experience collaborating with designers and backend engineers to deliver end-to-end web solutions.

Proficient in analyzing structured datasets using Python and SQL to identify trends, patterns, and performance metrics. Regularly performs data cleaning, exploratory analysis, and visualization to support reporting and decision-making. Experienced in creating dashboards and summaries that communicate insights clearly to technical and non-technical stakeholders.

Worked on projects that combine web development with data analysis, such as embedding analytical charts and reports into web applications. Experience handling user interaction data, monitoring usage metrics, and improving application features based on analytical findings.

Strong problem-solving skills with the ability to balance technical implementation and analytical reasoning. Interested in roles that blend web development and data analysis to create data-informed digital products."""]
top5_jobs = guess_top5_jobs(sample_resume_2)
top5_jobs

array(['ETL Developer', 'Data Science', 'Operations Manager',
       'SAP Developer', 'Java Developer'], dtype=object)